# Machine Learning Competition: YouTube Video Trend Prediction

Welcome to the challenge! In this notebook, you will build a machine learning model to predict whether a YouTube video will become a major trend and surpass 1 million views (`high_views` = 1) or not (`high_views` = 0).

## Core Requirements & Guidelines:

1. No Data Leakage: All future engagement metrics (views, likes, dislikes, comment counts) have been completely removed from your training and testing features. You must predict success based purely on pre-publish metadata.

2. Text Columns: Since we have not covered Advanced Natural Language Processing (NLP) yet, you should drop unstructured text columns like title and description from your training set, or extract simple numerical statistics (such as the character length of the title).

3. Feature Engineering: Look closely at the remaining columns in your dataset. Try to think of creative ways to transform or decompose raw data fields into meaningful numerical or categorical features that could give your model a competitive edge.

4. Mandatory Pipeline Architecture: You are strictly encouraged to use sklearn.pipeline.Pipeline and sklearn.compose.ColumnTransformer. Group your categorical encoders (e.g., OneHotEncoder or OrdinalEncoder) and numerical scalers together with your chosen classifier. This prevents leakage during validation and ensures production-grade code structure.

In [53]:
# Step 1: Import necessary libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeClassifier


## Step 2: Data Loading & Initial Inspection

Load the training dataset and perform basic exploratory data analysis using `head()`, `info()`, and checking for null values.

In [54]:
# Load the training data
train_df = pd.read_csv('USvideos_train.csv')
test_df  = pd.read_csv('USvideos_test_features.csv')

# Display the first 5 rows
train_df.head(15)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,high_views
0,YpUDblEURdY,18.02.06,We Tried To Make The Most Intimidating Pasta D...,Tasty,26,2018-05-26T15:00:09.000Z,"BuzzFeed|""BuzzFeed Tasty""|""Tasty""|""pasta""|""tim...",https://i.ytimg.com/vi/YpUDblEURdY/default.jpg,False,False,False,“A little danger adds a lot of excitement to a...,1
1,-kTWIm7m4Uw,18.30.05,We Bought A House,JennaMarbles,23,2018-05-16T22:33:29.000Z,"jenna|""marbles""|""mourey""|""we""|""bought""|""a hous...",https://i.ytimg.com/vi/-kTWIm7m4Uw/default.jpg,False,False,False,This house means the world to us. The absolut...,1
2,qVWzDX5NrhQ,18.02.03,I BOUGHT A CARDBOARD CUTOUT OF MYSELF,jennxpenn,24,2018-02-21T21:18:17.000Z,"jennxpenn|""jennxpenn2""|""jennxpenngames""|""jenn""...",https://i.ytimg.com/vi/qVWzDX5NrhQ/default.jpg,False,False,False,I make and buy a cardboard cutout of myself!\n...,0
3,L29S0JPOrV0,18.01.05,BEYONCE'S MAKEUP ARTIST DOES MY MAKEUP! | ft. ...,NikkieTutorials,26,2018-04-15T19:44:18.000Z,"beyonce|""beyonce's makeup artist does my makeu...",https://i.ytimg.com/vi/L29S0JPOrV0/default.jpg,False,False,False,After Beyonce’s ICONIC performance at Coachell...,1
4,P6Ys3PMJ3WY,18.07.02,I Like Beer (:60) | Michelob ULTRA Super Bowl,Michelob ULTRA,17,2018-01-31T14:16:10.000Z,"Michelob ULTRA|""Golf""|""Cycling""|""ultra light b...",https://i.ytimg.com/vi/P6Ys3PMJ3WY/default.jpg,False,True,False,When you love the superior taste of Michelob U...,1
5,ooyjaVdt-jA,18.28.05,Official Call of Duty®: Black Ops 4 — Multipla...,Call of Duty,20,2018-05-17T17:09:38.000Z,"call of duty|""cod""|""activision""|""Black Ops 4""",https://i.ytimg.com/vi/ooyjaVdt-jA/default.jpg,False,False,False,Call of Duty: Black Ops 4 Multiplayer raises t...,1
6,e0hO5fTGwPY,17.29.12,U.S. v. Whistleblower Tom Drake,CBS News,25,2011-05-22T23:50:01.000Z,"60 minutes|""us""|""whistleblower""|""tom""|""drake""|...",https://i.ytimg.com/vi/e0hO5fTGwPY/default.jpg,False,False,False,"Tom Drake, a former NSA senior executive indic...",0
7,2nVPFdqejD0,18.13.06,"Wait, My Crush is Watching? | Hannah Stocking ...",Hannah Stocking,23,2018-05-31T16:00:01.000Z,"wait my crush is watching|""hannah""|""stocking""|...",https://i.ytimg.com/vi/2nVPFdqejD0/default.jpg,False,False,False,WATCH MORE ▶ https://youtu.be/2-tzlBFotjASUBSC...,1
8,jsnyz__pc5M,18.12.02,Chloe x Halle - The Kids Are Alright - Officia...,Chloe x Halle,10,2018-02-09T16:14:59.000Z,"Chloe x Halle|""Chloe and Halle""|""Chloe and Hal...",https://i.ytimg.com/vi/jsnyz__pc5M/default.jpg,False,False,False,The official music video for our song “The Kid...,0
9,y_WoOYybCro,18.15.05,How I Became The Fresh Prince of Bel-Air | STO...,Will Smith,24,2018-05-10T16:08:58.000Z,"comedy|""entertainment""|""will smith""|""will""|""sm...",https://i.ytimg.com/vi/y_WoOYybCro/default.jpg,False,False,False,"The full story of how an ex-girlfriend, Arseni...",1


In [55]:
# Check dataset metadata, shapes, and structural data types
print("--- Dataset Information ---")
train_df.info()

--- Dataset Information ---
<class 'pandas.DataFrame'>
RangeIndex: 32759 entries, 0 to 32758
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   video_id                32759 non-null  str  
 1   trending_date           32759 non-null  str  
 2   title                   32759 non-null  str  
 3   channel_title           32759 non-null  str  
 4   category_id             32759 non-null  int64
 5   publish_time            32759 non-null  str  
 6   tags                    32759 non-null  str  
 7   thumbnail_link          32759 non-null  str  
 8   comments_disabled       32759 non-null  bool 
 9   ratings_disabled        32759 non-null  bool 
 10  video_error_or_removed  32759 non-null  bool 
 11  description             32303 non-null  str  
 12  high_views              32759 non-null  int64
dtypes: bool(3), int64(2), str(8)
memory usage: 2.6 MB


In [56]:
# Check for missing or null values inside columns
print("--- Missing Values Summary ---")
print(train_df.isnull().sum())

--- Missing Values Summary ---
video_id                    0
trending_date               0
title                       0
channel_title               0
category_id                 0
publish_time                0
tags                        0
thumbnail_link              0
comments_disabled           0
ratings_disabled            0
video_error_or_removed      0
description               456
high_views                  0
dtype: int64


## Step 3: Feature Engineering

In this section, perform any feature engineering, handling of missing values, or data transformations.
Write your custom transformation logic below:.

In [57]:
# Prepare your final training features and process the columns
new_train_df = train_df.copy()
new_train_df.drop(['thumbnail_link'], axis=1, inplace=True)
new_train_df['publish_time'] = pd.to_datetime(new_train_df['publish_time'])
new_train_df.head()

C:\Users\mohammad\AppData\Local\Temp\ipykernel_4812\3739208867.py:3: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  new_train_df.drop(['thumbnail_link'], axis=1, inplace=True)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,comments_disabled,ratings_disabled,video_error_or_removed,description,high_views
0,YpUDblEURdY,18.02.06,We Tried To Make The Most Intimidating Pasta D...,Tasty,26,2018-05-26 15:00:09+00:00,"BuzzFeed|""BuzzFeed Tasty""|""Tasty""|""pasta""|""tim...",False,False,False,“A little danger adds a lot of excitement to a...,1
1,-kTWIm7m4Uw,18.30.05,We Bought A House,JennaMarbles,23,2018-05-16 22:33:29+00:00,"jenna|""marbles""|""mourey""|""we""|""bought""|""a hous...",False,False,False,This house means the world to us. The absolut...,1
2,qVWzDX5NrhQ,18.02.03,I BOUGHT A CARDBOARD CUTOUT OF MYSELF,jennxpenn,24,2018-02-21 21:18:17+00:00,"jennxpenn|""jennxpenn2""|""jennxpenngames""|""jenn""...",False,False,False,I make and buy a cardboard cutout of myself!\n...,0
3,L29S0JPOrV0,18.01.05,BEYONCE'S MAKEUP ARTIST DOES MY MAKEUP! | ft. ...,NikkieTutorials,26,2018-04-15 19:44:18+00:00,"beyonce|""beyonce's makeup artist does my makeu...",False,False,False,After Beyonce’s ICONIC performance at Coachell...,1
4,P6Ys3PMJ3WY,18.07.02,I Like Beer (:60) | Michelob ULTRA Super Bowl,Michelob ULTRA,17,2018-01-31 14:16:10+00:00,"Michelob ULTRA|""Golf""|""Cycling""|""ultra light b...",False,True,False,When you love the superior taste of Michelob U...,1


In [58]:
new_train_df['tags'].fillna('[none]', inplace=True)

new_train_df['tag_number'] = np.where(new_train_df['tags'] == '[none]', 0, new_train_df['tags'].str.count(r'\|') + 1)
new_train_df[["tags", "tag_number"]].head(15)

C:\Users\mohammad\AppData\Local\Temp\ipykernel_4812\3865495002.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/docs/dev/user_guide/migration.html#copy-on-write-cow
  new_train_df['tags'].fillna('[none]', inplace=True)


,tags,tag_number
0,"BuzzFeed|""BuzzFeed Tasty""|""Tasty""|""pasta""|""tim...",11
1,"jenna|""marbles""|""mourey""|""we""|""bought""|""a hous...",41
2,"jennxpenn|""jennxpenn2""|""jennxpenngames""|""jenn""...",34
3,"beyonce|""beyonce's makeup artist does my makeu...",39
4,"Michelob ULTRA|""Golf""|""Cycling""|""ultra light b...",13
5,"call of duty|""cod""|""activision""|""Black Ops 4""",4
6,"60 minutes|""us""|""whistleblower""|""tom""|""drake""|...",15
7,"wait my crush is watching|""hannah""|""stocking""|...",26
8,"Chloe x Halle|""Chloe and Halle""|""Chloe and Hal...",20
9,"comedy|""entertainment""|""will smith""|""will""|""sm...",25


In [59]:
new_train_df["comments_disabled"] = new_train_df["comments_disabled"].astype(int)
new_train_df["ratings_disabled"] = new_train_df["ratings_disabled"].astype(int)
new_train_df["video_error_or_removed"] = new_train_df["video_error_or_removed"].astype(int)
new_train_df["trending_date"] = pd.to_datetime(new_train_df["trending_date"], format='%y.%d.%m')

In [60]:
pub = pd.to_datetime(new_train_df['publish_time'], utc=True).dt.tz_localize(None)
td = pd.to_datetime(new_train_df['trending_date'], format='%y.%d.%m')

new_train_df['_day'] = (td - pd.Timestamp('2017-01-01')).dt.days
new_train_df['days_to_trend'] = (td - pub.dt.normalize()).dt.days
new_train_df.head(15)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,comments_disabled,ratings_disabled,video_error_or_removed,description,high_views,tag_number,_day,days_to_trend
0,YpUDblEURdY,2018-06-02,We Tried To Make The Most Intimidating Pasta D...,Tasty,26,2018-05-26 15:00:09+00:00,"BuzzFeed|""BuzzFeed Tasty""|""Tasty""|""pasta""|""tim...",0,0,0,“A little danger adds a lot of excitement to a...,1,11,517,7
1,-kTWIm7m4Uw,2018-05-30,We Bought A House,JennaMarbles,23,2018-05-16 22:33:29+00:00,"jenna|""marbles""|""mourey""|""we""|""bought""|""a hous...",0,0,0,This house means the world to us. The absolut...,1,41,514,14
2,qVWzDX5NrhQ,2018-03-02,I BOUGHT A CARDBOARD CUTOUT OF MYSELF,jennxpenn,24,2018-02-21 21:18:17+00:00,"jennxpenn|""jennxpenn2""|""jennxpenngames""|""jenn""...",0,0,0,I make and buy a cardboard cutout of myself!\n...,0,34,425,9
3,L29S0JPOrV0,2018-05-01,BEYONCE'S MAKEUP ARTIST DOES MY MAKEUP! | ft. ...,NikkieTutorials,26,2018-04-15 19:44:18+00:00,"beyonce|""beyonce's makeup artist does my makeu...",0,0,0,After Beyonce’s ICONIC performance at Coachell...,1,39,485,16
4,P6Ys3PMJ3WY,2018-02-07,I Like Beer (:60) | Michelob ULTRA Super Bowl,Michelob ULTRA,17,2018-01-31 14:16:10+00:00,"Michelob ULTRA|""Golf""|""Cycling""|""ultra light b...",0,1,0,When you love the superior taste of Michelob U...,1,13,402,7
5,ooyjaVdt-jA,2018-05-28,Official Call of Duty®: Black Ops 4 — Multipla...,Call of Duty,20,2018-05-17 17:09:38+00:00,"call of duty|""cod""|""activision""|""Black Ops 4""",0,0,0,Call of Duty: Black Ops 4 Multiplayer raises t...,1,4,512,11
6,e0hO5fTGwPY,2017-12-29,U.S. v. Whistleblower Tom Drake,CBS News,25,2011-05-22 23:50:01+00:00,"60 minutes|""us""|""whistleblower""|""tom""|""drake""|...",0,0,0,"Tom Drake, a former NSA senior executive indic...",0,15,362,2413
7,2nVPFdqejD0,2018-06-13,"Wait, My Crush is Watching? | Hannah Stocking ...",Hannah Stocking,23,2018-05-31 16:00:01+00:00,"wait my crush is watching|""hannah""|""stocking""|...",0,0,0,WATCH MORE ▶ https://youtu.be/2-tzlBFotjASUBSC...,1,26,528,13
8,jsnyz__pc5M,2018-02-12,Chloe x Halle - The Kids Are Alright - Officia...,Chloe x Halle,10,2018-02-09 16:14:59+00:00,"Chloe x Halle|""Chloe and Halle""|""Chloe and Hal...",0,0,0,The official music video for our song “The Kid...,0,20,407,3
9,y_WoOYybCro,2018-05-15,How I Became The Fresh Prince of Bel-Air | STO...,Will Smith,24,2018-05-10 16:08:58+00:00,"comedy|""entertainment""|""will smith""|""will""|""sm...",0,0,0,"The full story of how an ex-girlfriend, Arseni...",1,25,499,5


In [61]:
target = new_train_df["high_views"]

for col in ["channel_title", "video_id"]:
    encoder = TargetEncoder(
        target_type="binary",
        random_state=42
    )

    new_train_df[col] = encoder.fit_transform(
        new_train_df[[col]],
        target
    ).ravel()


c:\Users\mohammad\Tamkeen\7-Machine_learning\.venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\mohammad\Tamkeen\7-Machine_learning\.venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


In [62]:
new_train_df["category_id"] = new_train_df["category_id"].astype(str)

category_dummies = pd.get_dummies(
    new_train_df["category_id"],
    prefix="category_id",
    dtype=float
)

new_train_df = pd.concat(
    [
        new_train_df.drop(columns=["category_id"]),
        category_dummies
    ],
    axis=1
)

In [63]:

from pandas import to_datetime

Periods = {
    "morning": to_datetime("06:00:00").time(),
    "afternoon": to_datetime("12:00:00").time(),
    "evening": to_datetime("18:00:00").time(),
    "midnight": to_datetime("00:00:00").time(),
}
def get_period(time):
    if time >= Periods["midnight"] and time < Periods["morning"]:
        return "midnight"
    elif time >= Periods["morning"] and time < Periods["afternoon"]:
        return "morning"
    elif time >= Periods["afternoon"] and time < Periods["evening"]:
        return "afternoon"
    else:
        return "evening"


new_train_df["publish_time_period"] = new_train_df["publish_time"].dt.time.apply(get_period)
new_train_df["publish_time_period_number"] = new_train_df["publish_time_period"].map({"midnight": 0, "morning": 1, "afternoon": 2, "evening": 3})
new_train_df.head(15)

,video_id,trending_date,title,channel_title,publish_time,tags,comments_disabled,ratings_disabled,video_error_or_removed,description,...,category_id_23,category_id_24,category_id_25,category_id_26,category_id_27,category_id_28,category_id_29,category_id_43,publish_time_period,publish_time_period_number
0,0.752803,2018-06-02,We Tried To Make The Most Intimidating Pasta D...,0.617007,2018-05-26 15:00:09+00:00,"BuzzFeed|""BuzzFeed Tasty""|""Tasty""|""pasta""|""tim...",0,0,0,“A little danger adds a lot of excitement to a...,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,afternoon,2
1,1.000000,2018-05-30,We Bought A House,1.000000,2018-05-16 22:33:29+00:00,"jenna|""marbles""|""mourey""|""we""|""bought""|""a hous...",0,0,0,This house means the world to us. The absolut...,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,evening,3
2,0.000000,2018-03-02,I BOUGHT A CARDBOARD CUTOUT OF MYSELF,0.000000,2018-02-21 21:18:17+00:00,"jennxpenn|""jennxpenn2""|""jennxpenngames""|""jenn""...",0,0,0,I make and buy a cardboard cutout of myself!\n...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,evening,3
3,1.000000,2018-05-01,BEYONCE'S MAKEUP ARTIST DOES MY MAKEUP! | ft. ...,0.985223,2018-04-15 19:44:18+00:00,"beyonce|""beyonce's makeup artist does my makeu...",0,0,0,After Beyonce’s ICONIC performance at Coachell...,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,evening,3
4,0.000000,2018-02-07,I Like Beer (:60) | Michelob ULTRA Super Bowl,0.399846,2018-01-31 14:16:10+00:00,"Michelob ULTRA|""Golf""|""Cycling""|""ultra light b...",0,1,0,When you love the superior taste of Michelob U...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,afternoon,2
5,1.000000,2018-05-28,Official Call of Duty®: Black Ops 4 — Multipla...,1.000000,2018-05-17 17:09:38+00:00,"call of duty|""cod""|""activision""|""Black Ops 4""",0,0,0,Call of Duty: Black Ops 4 Multiplayer raises t...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,afternoon,2
6,0.000000,2017-12-29,U.S. v. Whistleblower Tom Drake,0.282424,2011-05-22 23:50:01+00:00,"60 minutes|""us""|""whistleblower""|""tom""|""drake""|...",0,0,0,"Tom Drake, a former NSA senior executive indic...",...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,evening,3
7,1.000000,2018-06-13,"Wait, My Crush is Watching? | Hannah Stocking ...",0.860205,2018-05-31 16:00:01+00:00,"wait my crush is watching|""hannah""|""stocking""|...",0,0,0,WATCH MORE ▶ https://youtu.be/2-tzlBFotjASUBSC...,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,afternoon,2
8,0.000000,2018-02-12,Chloe x Halle - The Kids Are Alright - Officia...,0.000000,2018-02-09 16:14:59+00:00,"Chloe x Halle|""Chloe and Halle""|""Chloe and Hal...",0,0,0,The official music video for our song “The Kid...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,afternoon,2
9,1.000000,2018-05-15,How I Became The Fresh Prince of Bel-Air | STO...,0.661891,2018-05-10 16:08:58+00:00,"comedy|""entertainment""|""will smith""|""will""|""sm...",0,0,0,"The full story of how an ex-girlfriend, Arseni...",...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,afternoon,2


In [ ]:
def add_previous_next_features(data: pd.DataFrame, training_data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    data["prev_label"] = np.nan
    data["next_label"] = np.nan
    data["prev_gap"] = np.nan
    data["next_gap"] = np.nan

    for index, row in data.iterrows():

        same_video = training_data[
            training_data["video_id"] == row["video_id"]
        ]

        earlier = same_video[
            same_video["_day"] < row["_day"]
        ]

        if len(earlier) > 0:
            previous_row = earlier.loc[
                earlier["_day"].idxmax()
            ]

            data.loc[index, "prev_label"] = previous_row["high_views"]
            data.loc[index, "prev_gap"] = (
                row["_day"] - previous_row["_day"]
            )

        later = same_video[
            same_video["_day"] > row["_day"]
        ]

        if len(later) > 0:
            next_row = later.loc[
                later["_day"].idxmin()
            ]

            data.loc[index, "next_label"] = next_row["high_views"]
            data.loc[index, "next_gap"] = (
                next_row["_day"] - row["_day"]
            )

    return data

In [68]:
features = [
    "channel_title",
    "video_id",
    "tag_number",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "publish_time_period_number",
    "days_to_trend",
    "_day",
]

features += list(category_dummies.columns)

X = new_train_df[features]
y = new_train_df["high_views"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_rows = X_train.copy()
train_rows["high_views"] = y_train.values

X_train = add_previous_next_features(X_train, train_rows)
X_test = add_previous_next_features(X_test, train_rows)

features += [
    "prev_label",
    "next_label",
    "prev_gap",
    "next_gap",
]

X_train = X_train[features]
X_test = X_test[features]

X_train["prev_label"] = X_train["prev_label"].fillna(-1)
X_test["prev_label"] = X_test["prev_label"].fillna(-1)

X_train["next_label"] = X_train["next_label"].fillna(-1)
X_test["next_label"] = X_test["next_label"].fillna(-1)

X_train["prev_gap"] = X_train["prev_gap"].fillna(999)
X_test["prev_gap"] = X_test["prev_gap"].fillna(999)

X_train["next_gap"] = X_train["next_gap"].fillna(999)
X_test["next_gap"] = X_test["next_gap"].fillna(999)

In [69]:
knn_baseline = KNeighborsClassifier(n_neighbors=15)
knn_baseline.fit(X_train, y_train)
y_pred = knn_baseline.predict(X_test)
knn_f1 = f1_score(y_test, y_pred)
print(f"KNN F1 Score: {knn_f1:.4f}")

KNN F1 Score: 0.7432


In [70]:

scaler = StandardScaler()

X_train_scaled  = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)



knn_scaled = KNeighborsClassifier(n_neighbors=15)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
knn_f1 = f1_score(y_test, y_pred_scaled)
print(f"KNN (SCALED) F1 Score: {knn_f1:.4f}")

KNN (SCALED) F1 Score: 0.9223


In [71]:
X_train.head(15)

,channel_title,video_id,tag_number,comments_disabled,ratings_disabled,video_error_or_removed,publish_time_period_number,days_to_trend,_day,category_id_1,...,category_id_25,category_id_26,category_id_27,category_id_28,category_id_29,category_id_43,prev_label,next_label,prev_gap,next_gap
5063,0.000000,0.000000,10,0,0,0,2,4,518,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
9172,0.000000,0.000000,0,0,0,0,2,1520,377,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
27888,0.000000,0.000000,26,0,0,0,3,9,369,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
21939,0.000000,0.000000,22,0,0,0,3,13,450,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
31790,0.000000,0.000000,6,0,0,0,0,4,388,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
11782,0.499101,0.000000,27,0,0,0,1,4,352,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
12269,0.955436,1.000000,33,0,0,0,3,14,460,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
14933,0.307978,0.479139,12,0,0,0,2,7,391,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,5.0,10.0
26346,0.000000,0.000000,50,0,0,0,3,1,332,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
27081,0.000000,0.000000,22,0,0,0,1,8,363,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [72]:
ds = DecisionTreeClassifier(random_state=42, criterion="entropy", max_depth=10, min_samples_leaf=5)
ds.fit(X_train, y_train)
y_pred = ds.predict(X_test)
ds_f1 = f1_score(y_test, y_pred)
print(f"Decision Tree F1 Score: {ds_f1:.4f}")

Decision Tree F1 Score: 0.9321


In [73]:
from sklearn.linear_model import LogisticRegression


lr = LogisticRegression(random_state=42, max_iter=2000)
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
lr_f1 = f1_score(y_test, y_pred)
print(f"Logistic Regression F1 Score: {lr_f1:.4f}")

Logistic Regression F1 Score: 0.9228


In [74]:
rf = RandomForestClassifier(random_state=42,oob_score=True, n_estimators=200, max_depth=5, min_samples_split=2, min_samples_leaf=2)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
rf_f1 = f1_score(y_test, y_pred)
print(f"Random Forest F1 Score: {rf_f1:.4f}")

Random Forest F1 Score: 0.9324


In [77]:
important = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Feature Importances:")
important

Feature Importances:


video_id                      0.324235
next_label                    0.238553
prev_label                    0.225153
channel_title                 0.130976
days_to_trend                 0.028385
_day                          0.020421
next_gap                      0.012811
prev_gap                      0.011925
category_id_10                0.002315
category_id_25                0.001471
tag_number                    0.000915
category_id_1                 0.000728
category_id_26                0.000378
category_id_27                0.000313
publish_time_period_number    0.000247
category_id_23                0.000246
category_id_20                0.000215
category_id_15                0.000156
category_id_28                0.000121
category_id_24                0.000101
category_id_19                0.000092
category_id_2                 0.000055
category_id_17                0.000051
comments_disabled             0.000049
ratings_disabled              0.000037
category_id_22           

In [78]:
from sklearn.ensemble import VotingClassifier


hard_voting = VotingClassifier(estimators=[('knn', knn_scaled), ('dt', ds), ('lr', lr), ('rf', rf)], voting='hard')
hard_voting.fit(X_train, y_train)
y_pred = hard_voting.predict(X_test)
hard_voting_f1 = f1_score(y_test, y_pred)
print(f"Hard Voting Classifier F1 Score: {hard_voting_f1:.4f}")

Hard Voting Classifier F1 Score: 0.9287


In [80]:


from sklearn.ensemble import StackingClassifier


base_learners = [
    ('knn', knn_scaled),
    ('dt', ds),
    ('lr', lr),
    ('rf', rf),
]
meta_learner = LogisticRegression(random_state=42, max_iter=4000)

stacker = StackingClassifier(estimators=base_learners, final_estimator=meta_learner, cv=5, passthrough=True, n_jobs=-1)

stacker.fit(X_train, y_train)
y_pred = stacker.predict(X_test)
stacker_f1 = f1_score(y_test, y_pred)
print(f"Stacking Classifier F1 Score: {stacker_f1:.4f}")

Stacking Classifier F1 Score: 0.9409


In [81]:
from sklearn.model_selection import GridSearchCV


param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None,10, 25],
    "min_samples_leaf": [1, 5, 20],
    'criterion': ['gini', 'entropy'],
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    refit=True,
)

grid_search.fit(X_train, y_train)
gy_pred = grid_search.predict(X_test)
grid_search_f1 = f1_score(y_test, gy_pred)
print(f"Grid Search best parameters: {grid_search.best_params_}")
print(f"Grid Search KNN F1 Score: {grid_search_f1:.4f}") # was 68%

Grid Search best parameters: {'criterion': 'gini', 'max_depth': 25, 'min_samples_leaf': 1, 'n_estimators': 100}
Grid Search KNN F1 Score: 0.9450


In [82]:


param_grid = {
    "n_neighbors": [5, 15 ,31],
    "weights": ["uniform", "distance"],
}

grid_search = GridSearchCV(
    estimator=knn_scaled,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    refit=True,
)

grid_search.fit(X_train_scaled, y_train)
gy_pred = grid_search.predict(X_test_scaled)
grid_search_f1 = f1_score(y_test, gy_pred)
print(f"Grid Search best parameters: {grid_search.best_params_}")
print(f"Grid Search KNN F1 Score: {grid_search_f1:.4f}") # was 66%

Grid Search best parameters: {'n_neighbors': 15, 'weights': 'distance'}
Grid Search KNN F1 Score: 0.9259


## Step 4: Performance Evaluation Helper Function

Use this comprehensive evaluation tool to print standard classification metrics and inspect the resulting confusion matrix on your local validation subsets.

In [ ]:
def evaluate_model(y_true, y_pred):
    """
    Computes standard classification metrics and plots a visual Confusion Matrix.
    """
    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    matrix = confusion_matrix(y_true, y_pred)
    
    print("=========================================")
    print("         Model Performance Evaluation     ")
    print("=========================================")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("=========================================")
    

    disp = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=['Not Trend', 'Trend'])
    
    disp.plot(cmap=plt.cm.Blues, values_format='d')
    plt.title("Confusion Matrix Heatmap")
    plt.show()
    
    return {
        "accuracy": accuracy,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": matrix
    }

## Step 5: Model Pipeline & Validation

Split your data locally into train/validation sets, construct your preprocessing wrappers, build a pipeline, and fit your model.

In [7]:
# Define features and target label
# y = train_df['high_views']
# X = train_df.drop(columns=['high_views', ...])

# Build and fit your Pipeline infrastructure below:

## Step 6: Test Predictions & Submission File Generation

When you are satisfied with your local validation F1-score, load the unlabeled testing features, compute final predictions using your pipeline, and run the submission script below to save your predictions.

In [ ]:
def submit_predictions(test_predictions, team_name):
    """
    Saves the final test predictions to a clean CSV format for grading.
    
    Parameters:
    test_predictions: Flat array or list containing the predicted labels (0 or 1) for the test dataset.
    team_name: String specifying your team identifier.
    """
    predictions_list = list(test_predictions)
    
    # Prepare formatting
    submission_df = pd.DataFrame({
        'high_views': predictions_list
    })
    
    output_filename = f"submission_{team_name}.csv"
    
    # Save file without indices to match evaluation backend expectations
    submission_df.to_csv(output_filename, index=False)
    
    print("=========================================")
    print("STATUS: Submission file successfully created")
    print(f"Output Filename : {output_filename}")
    print(f"Total Row Count : {len(submission_df)}")
    print("Action Required : Submit this generated file into the evaluation platform.")
    print("=========================================")

# Usage Example:
test_features_df = pd.read_csv('USvideos_test_features.csv')
final_preds = predict(test_features_df)
submit_predictions(final_preds, team_name="Team_Name_Here")

## Submission link
https://youtubechallenge-9o6qyy9apphdrfpcxvvw7qj.streamlit.app/

## (Optional Bonus): Hyperparameter Tuning
If your baseline model is working and you want to improve your F1-Score, 
you can optionally use GridSearchCV or RandomizedSearchCV here.

Note: Keep your parameter grid small to avoid long training times during the competition.